# pKa Data Preparation

## Notebook Contract
- Input file: `./Datasets/Raw Dataset/OriginalDatapKa.csv`
- Required columns: `SMILES`, `pka_type`, `T`, `pka_value`
- Scope: baseline 25 C workflow by default, with an optional temperature-feature mode for all `pKa1` rows that have numeric `T`; when temperature mode is on, duplicate `SMILES` are allowed only if `T` differs
- Outputs:
  - Curated dataset: `./Datasets/Curated Dataset(s)/pka_data_clean.csv`
  - Chemprop trainable file: `./Datasets/trainable_data/pka_chemprop_train_file.csv`
  - Optional temperature features: `./Datasets/trainable_data/pka_chemprop_feat_file.csv` and `./Datasets/trainable_data/pka_chemprop_feat_file.npz`


## 1. Setup and Load Raw Data


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from rdkit import Chem
from sklearn.model_selection import train_test_split

# Configuration
RAW_PATH = './Datasets/Raw Dataset/OriginalDatapKa.csv'
CURATED_DIR = './Datasets/Curated Dataset(s)'
TRAINABLE_DIR = './Datasets/trainable_data'
CURATED_FILENAME = 'pka_data_clean.csv'
TRAINABLE_FILENAME = 'pka_chemprop_train_file.csv'
FEATURE_FILENAME = 'pka_chemprop_feat_file.csv'
FEATURE_NPZ_FILENAME = 'pka_chemprop_feat_file.npz'
USE_TEMPERATURE_FEATURES = True
TEMP_C = 25
RANDOM_SEED = 42
MAX_HEAVY_ATOMS = 40
TEMP_HIST_BIN_WIDTH = 2

# Load raw data
raw_df = pd.read_csv(RAW_PATH)

# Validate required schema early
required_cols = {'SMILES', 'pka_type', 'T', 'pka_value'}
missing_cols = required_cols - set(raw_df.columns)
if missing_cols:
    raise ValueError(f'Missing required columns: {sorted(missing_cols)}')

print(f'Loaded raw rows: {len(raw_df)}')


## 2. Canonicalize SMILES


In [ ]:
# Canonicalize SMILES with RDKit before filtering

def canonicalize_smiles(smi):
    if pd.isna(smi):
        return smi
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return smi
    return Chem.MolToSmiles(mol, canonical=True)

df = raw_df.copy()
original_smiles = df['SMILES'].copy()
df['SMILES'] = df['SMILES'].apply(canonicalize_smiles)

changed = (original_smiles != df['SMILES']).sum()
invalid = df['SMILES'].apply(lambda s: Chem.MolFromSmiles(str(s)) is None if pd.notna(s) else False).sum()

print(f'Canonicalized SMILES entries changed: {changed}')
print(f'Entries still non-parseable by RDKit after canonicalization: {invalid}')
print(f'Rows after canonicalization step: {len(df)}')


## 3. Build Baseline or Temperature-Aware Dataset


In [ ]:
# Build finalized pKa training data with an optional temperature feature
df['T'] = pd.to_numeric(df['T'], errors='coerce')

start_rows = len(df)
pka1_df = df[df['pka_type'] == 'pKa1'].copy()

if USE_TEMPERATURE_FEATURES:
    numeric_t_df = pka1_df[pka1_df['T'].notna()].copy()
    numeric_t_df['T_C'] = numeric_t_df['T'].astype(float)
    baseline_df = numeric_t_df.drop_duplicates(subset=['SMILES', 'T_C'], keep='first').copy()
    pka_data = baseline_df[['SMILES', 'pka_value', 'T_C']].copy()
else:
    t25_df = pka1_df[pka1_df['T'] == TEMP_C].copy()
    baseline_df = t25_df.drop_duplicates(subset='SMILES', keep='first').copy()
    pka_data = baseline_df[['SMILES', 'pka_value']].copy()

df = pka_data.copy()

if pka_data.empty:
    raise ValueError('Baseline dataset is empty after filtering. Check source data and filters.')

print('Baseline filter breakdown:')
print(f'- Start rows: {start_rows}')
print(f'- After pKa1 filter: {len(pka1_df)}')
if USE_TEMPERATURE_FEATURES:
    print(f'- After numeric T filter: {len(numeric_t_df)}')
    print(f'- After SMILES + T_C deduplication: {len(pka_data)}')
    print(f'- Rows with missing T removed before feature export: {len(pka1_df) - len(numeric_t_df)}')
else:
    print(f'- After T == {TEMP_C} filter: {len(t25_df)}')
    print(f'- After SMILES deduplication: {len(pka_data)}')


In [ ]:
def clean_smiles_dataset(df, output_path, threshold=0.01, max_heavy_atoms=40):
    """
    Cleans a dataset containing SMILES strings:
      - Canonicalizes SMILES
      - Removes invalid molecules
      - Removes inorganic molecules (no carbon)
      - Removes organometallic molecules
      - Removes ions
      - Drops duplicates
      - Removes molecules with elements appearing in < threshold fraction of dataset
      - Saves cleaned DataFrame as CSV with '_clean' suffix

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing a SMILES column ("SMILES" or "smiles").
    output_path : str
        Path to save the cleaned CSV (suffix '_clean.csv' will be added automatically).
    threshold : float
        Minimum fraction of molecules required for an element to be kept (default=0.01 = 1%).

    Returns
    -------
    cleaned_df : pd.DataFrame
        The cleaned DataFrame.
    atom_summary : pd.DataFrame
        Summary of atom counts across dataset.
    """
    smiles_col = None
    for candidate in ["SMILES", "smiles"]:
        if candidate in df.columns:
            smiles_col = candidate
            break
    if smiles_col is None:
        raise ValueError("No 'SMILES' or 'smiles' column found in DataFrame.")

    def canonicalize_smiles(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None
            return Chem.MolToSmiles(mol, canonical=True)
        except Exception:
            return None

    def is_organic(mol):
        return any(atom.GetSymbol() == "C" for atom in mol.GetAtoms())

    def is_organometallic(mol):
        metal_atomic_numbers = {
            3,4,11,12,13,14,19,20,21,22,23,24,25,26,27,28,29,30,
            31,34,37,38,39,40,41,42,43,44,45,46,47,48,49,50,
            55,56,57,72,73,74,75,76,77,78,79,80,81,82,83,
            87,88,89
        }
        return any(atom.GetAtomicNum() in metal_atomic_numbers for atom in mol.GetAtoms())

    def is_ion(mol):
        return Chem.GetFormalCharge(mol) != 0

    def has_isotopic_label(mol):
        # Remove explicitly isotopically labeled compounds such as deuterated SMILES ([2H]).
        return any(atom.GetIsotope() > 0 for atom in mol.GetAtoms())


    def count_heavy_atoms(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return mol.GetNumHeavyAtoms()

    def analyze_elements(df_local, smiles_column):
        atom_type_counts = Counter()
        molecule_atom_presence = Counter()
        for smi in df_local[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            mol = Chem.AddHs(mol)
            atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
            atom_type_counts.update(atom_symbols)
            molecule_atom_presence.update(set(atom_symbols))
        return pd.DataFrame({
            "Total_Atom_Count": pd.Series(atom_type_counts),
            "Molecule_Count": pd.Series(molecule_atom_presence)
        }).fillna(0).astype(int)

    def remove_rare_elements(df_local, smiles_column, threshold_local):
        atom_summary_local = analyze_elements(df_local, smiles_column=smiles_column)
        total_molecules = len(df_local)
        if total_molecules == 0:
            return df_local.copy(), atom_summary_local, [], 0

        rare_elements = atom_summary_local[
            atom_summary_local["Molecule_Count"] / total_molecules < threshold_local
        ].index.tolist()
        mask = []
        for smi in df_local[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                mask.append(False)
                continue
            atoms = {atom.GetSymbol() for atom in mol.GetAtoms()}
            mask.append(not any(el in atoms for el in rare_elements))

        filtered_df = df_local[mask].reset_index(drop=True)
        removed_count = len(df_local) - len(filtered_df)
        return filtered_df, atom_summary_local, rare_elements, removed_count

    stats = {"invalid": [], "multi_component": [], "inorganic": [], "organometallic": [], "ion": [], "isotopic_label": [], "kept": []}

    for _, row in df.iterrows():
        row_copy = row.copy()
        smi = row_copy[smiles_col]
        cano = canonicalize_smiles(smi)
        if cano is None:
            stats["invalid"].append(row_copy)
            continue

        # Filter out multi-component SMILES (e.g., salts/mixtures like 'CCO.Cl')
        if '.' in str(cano):
            row_copy[smiles_col] = cano
            stats["multi_component"].append(row_copy)
            continue

        mol = Chem.MolFromSmiles(cano)
        if mol is None:
            stats["invalid"].append(row_copy)
            continue

        row_copy[smiles_col] = cano

        if not is_organic(mol):
            stats["inorganic"].append(row_copy)
            continue
        if is_organometallic(mol):
            stats["organometallic"].append(row_copy)
            continue
        if is_ion(mol):
            stats["ion"].append(row_copy)
            continue
        if has_isotopic_label(mol):
            stats["isotopic_label"].append(row_copy)
            continue
        stats["kept"].append(row_copy)

    pre_dedup_df = pd.DataFrame(stats["kept"])
    pre_dedup_count = len(pre_dedup_df)
    dedup_subset = [smiles_col]
    if 'T_C' in pre_dedup_df.columns:
        dedup_subset.append('T_C')
    dedup_df = pre_dedup_df.drop_duplicates(subset=dedup_subset, keep="first")
    dedup_removed = pre_dedup_count - len(dedup_df)

    cleaned_df, atom_summary, rare_elements, rare_removed = remove_rare_elements(dedup_df, smiles_col, threshold)

    cleaned_df['non_H_atom_count'] = cleaned_df[smiles_col].apply(count_heavy_atoms)
    heavy_before = len(cleaned_df)
    cleaned_df = cleaned_df[cleaned_df['non_H_atom_count'] <= max_heavy_atoms].copy()
    heavy_removed = heavy_before - len(cleaned_df)

    import inspect

    frame = inspect.currentframe().f_back
    df_name = None
    for var_name, var_val in frame.f_locals.items():
        if var_val is df:
            df_name = var_name
            break

    if df_name is None:
        df_name = "dataset"

    if not os.path.isdir(output_path):
        os.makedirs(output_path, exist_ok=True)

    output_file = os.path.join(output_path, f"{df_name}_clean.csv")
    cleaned_df.to_csv(output_file, index=False)
    print(f"Cleaned dataset saved to {output_file}")

    print("\nFilter breakdown (stepwise):")
    running = len(df)

    def print_step(label, removed):
        nonlocal running
        after = running - removed
        print(f"- {label}: removed {removed} | before {running} -> after {after}")
        running = after

    print_step("Invalid SMILES", len(stats["invalid"]))
    print_step("Multi-component SMILES (contains .)", len(stats["multi_component"]))
    print_step("Inorganic (no carbon)", len(stats["inorganic"]))
    print_step("Organometallic", len(stats["organometallic"]))
    print_step("Ions (non-zero formal charge)", len(stats["ion"]))
    print_step("Isotopically labeled SMILES", len(stats["isotopic_label"]))
    print_step("Duplicate SMILES", dedup_removed)
    print_step(f"Rare elements (<{threshold*100:.1f}% molecule frequency)", rare_removed)
    print_step(f"Heavy atom count > {max_heavy_atoms}", heavy_removed)

    if rare_elements:
        print(f"Rare elements identified: {rare_elements}")
    else:
        print("Rare elements identified: none")

    print(f"Final retained rows: {len(cleaned_df)}")

    return cleaned_df, atom_summary


In [ ]:
# Clean molecules and export curated dataset
os.makedirs(CURATED_DIR, exist_ok=True)
pka_df_clean, atom_summary = clean_smiles_dataset(
    pka_data,
    output_path=CURATED_DIR,
    max_heavy_atoms=MAX_HEAVY_ATOMS,
)

expected_curated_path = os.path.join(CURATED_DIR, CURATED_FILENAME)
print(f'Expected curated output path: {expected_curated_path}')


## 5. Split for Chemprop and Export Trainable File


In [ ]:
# Deterministic train/val/test split
if USE_TEMPERATURE_FEATURES:
    unique_smiles = pka_df_clean['SMILES'].drop_duplicates().reset_index(drop=True)
    train_smiles, temp_smiles = train_test_split(unique_smiles, test_size=0.2, random_state=RANDOM_SEED)
    val_smiles, test_smiles = train_test_split(temp_smiles, test_size=0.5, random_state=RANDOM_SEED)

    split_map = {smi: 'train' for smi in train_smiles}
    split_map.update({smi: 'val' for smi in val_smiles})
    split_map.update({smi: 'test' for smi in test_smiles})

    pka_df_clean['split'] = pka_df_clean['SMILES'].map(split_map)
    if pka_df_clean['split'].isna().any():
        missing_split = pka_df_clean[pka_df_clean['split'].isna()]['SMILES'].unique().tolist()
        raise ValueError(f'Failed to assign splits for SMILES: {missing_split[:5]}')

    train_df = pka_df_clean[pka_df_clean['split'] == 'train'].copy()
    val_df = pka_df_clean[pka_df_clean['split'] == 'val'].copy()
    test_df = pka_df_clean[pka_df_clean['split'] == 'test'].copy()
else:
    train_df, temp_df = train_test_split(pka_df_clean, test_size=0.2, random_state=RANDOM_SEED)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=RANDOM_SEED)

    train_df['split'] = 'train'
    val_df['split'] = 'val'
    test_df['split'] = 'test'

train_val_test_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

os.makedirs(TRAINABLE_DIR, exist_ok=True)
trainable_path = os.path.join(TRAINABLE_DIR, TRAINABLE_FILENAME)
train_export_columns = ['SMILES', 'pka_value', 'split']
train_val_test_df[train_export_columns].to_csv(trainable_path, index=False)

feat_csv_path = None
feat_npz_path = None
if USE_TEMPERATURE_FEATURES:
    feature_df = train_val_test_df[['SMILES', 'T_C', 'split']].copy()
    feat_csv_path = os.path.join(TRAINABLE_DIR, FEATURE_FILENAME)
    feature_df.to_csv(feat_csv_path, index=False)

    feat_npz_path = os.path.join(TRAINABLE_DIR, FEATURE_NPZ_FILENAME)
    np.savez(feat_npz_path, feature_df['T_C'].to_numpy(dtype=float).reshape(-1, 1))

print(f'Saved trainable dataset: {trainable_path}')
print(f"Split sizes -> train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")
if USE_TEMPERATURE_FEATURES:
    print(f"Unique SMILES per split -> train: {train_df['SMILES'].nunique()}, val: {val_df['SMILES'].nunique()}, test: {test_df['SMILES'].nunique()}")
if USE_TEMPERATURE_FEATURES:
    print(f'Saved feature CSV: {feat_csv_path}')
    print(f'Saved feature NPZ: {feat_npz_path}')
else:
    print('Temperature features disabled; no feature CSV/NPZ was written.')


## 6. Final Summary


In [ ]:
print('Run summary:')
print(f'- Raw rows loaded: {len(raw_df)}')
print(f'- Temperature-feature mode: {USE_TEMPERATURE_FEATURES}')
print(f'- Baseline rows retained (pre-clean): {len(pka_data)}')
print(f'- Curated rows retained: {len(pka_df_clean)}')
print(f"- Unique SMILES in curated set: {pka_df_clean['SMILES'].nunique()}")
print(f'- Trainable rows exported: {len(train_val_test_df)}')
print(f'- Curated CSV: {os.path.join(CURATED_DIR, CURATED_FILENAME)}')
print(f'- Trainable CSV: {os.path.join(TRAINABLE_DIR, TRAINABLE_FILENAME)}')
if USE_TEMPERATURE_FEATURES:
    print(f'- Feature CSV: {feat_csv_path}')
    print(f'- Feature NPZ: {feat_npz_path}')

if USE_TEMPERATURE_FEATURES and 'T_C' in pka_df_clean.columns:
    temp_values = pd.to_numeric(pka_df_clean['T_C'], errors='coerce').dropna().astype(float)
    if not temp_values.empty:
        q1 = temp_values.quantile(0.25)
        q3 = temp_values.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 10 * iqr
        upper_bound = q3 + 10 * iqr
        hist_temps = temp_values[(temp_values >= lower_bound) & (temp_values <= upper_bound)].copy()
        outlier_count = len(temp_values) - len(hist_temps)

        if hist_temps.empty:
            hist_temps = temp_values.copy()
            outlier_count = 0

        clipped_temps = hist_temps.clip(lower=TEMP_HIST_BIN_WIDTH)
        max_temp = float(np.ceil(clipped_temps.max() / TEMP_HIST_BIN_WIDTH) * TEMP_HIST_BIN_WIDTH)
        bins = np.arange(TEMP_HIST_BIN_WIDTH, max_temp + TEMP_HIST_BIN_WIDTH, TEMP_HIST_BIN_WIDTH)
        if bins.size < 2:
            bins = np.array([TEMP_HIST_BIN_WIDTH, TEMP_HIST_BIN_WIDTH * 2], dtype=float)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(clipped_temps, bins=bins, color='#4C72B0', edgecolor='white')
        ax.set_title('Temperature distribution in cleaned pKa dataset')
        ax.set_xlabel('Temperature (C)')
        ax.set_ylabel('Molecule count')
        ax.set_xlim(left=TEMP_HIST_BIN_WIDTH)
        ax.set_xticks(np.arange(TEMP_HIST_BIN_WIDTH, max_temp + TEMP_HIST_BIN_WIDTH, TEMP_HIST_BIN_WIDTH))
        ax.grid(axis='y', alpha=0.25)
        fig.tight_layout()
        temp_hist_path = os.path.join(TRAINABLE_DIR, 'pka_temperature_histogram.png')
        fig.savefig(temp_hist_path, dpi=200, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f'Temperature histogram saved to: {temp_hist_path}')
        print(f'- Removed {outlier_count} temperature outliers from the histogram using the 1.5*IQR rule.')
        print(f'- Temperatures below {TEMP_HIST_BIN_WIDTH} C were clipped into the first visible bin so they remain included.')
